In [2]:
# # Chocolate Dataset Creation

# This notebook builds a data-efficient chocolate dataset from:

# 1. Real photographs of trays containing known chocolate flavors
# 2. Limited photographs of filled chocolate boxes
# 3. Empty-slot references generated in `01_box_detection.ipynb`
# 4. Synthetic occupied-slot composites

# The final dataset will use source-image-grouped train, validation, and test splits.

In [5]:
!pwd
!ls

/homes/s959m963/Chocolathon
01_box_detection.ipynb		     Preprocessing.ipynb  Untitled.ipynb
02_chocolate_dataset_creation.ipynb  requirements.txt
data				     src


In [6]:
from pathlib import Path
from PIL import Image, ImageOps
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import cv2
import json
import os

def find_project_root():
    override = os.environ.get("CHOCOLATHON_ROOT")

    if override:
        root = Path(override).expanduser().resolve()
        if root.is_dir():
            return root
        raise FileNotFoundError(f"Invalid CHOCOLATHON_ROOT: {root}")

    start = Path.cwd().resolve()

    for candidate in [start, *start.parents]:
        if (
            candidate.name == "Chocolathon"
            and (candidate / "data").is_dir()
            and (candidate / "01_box_detection.ipynb").exists()
        ):
            return candidate

    raise FileNotFoundError(
        f"Could not locate Chocolathon from {start}. "
        "Open this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()

print("Project root:", PROJECT_ROOT)
print("Current directory:", Path.cwd().resolve())
print("Home directory:", Path.home())
print("OpenCV version:", cv2.__version__)

Project root: /homes/s959m963/Chocolathon
Current directory: /homes/s959m963/Chocolathon
Home directory: /homes/s959m963
OpenCV version: 4.13.0


In [7]:
BOX_DIR = PROJECT_ROOT / "data" / "boxes"
EMPTY_SLOTS_DIR = BOX_DIR / "slots" / "empty"
EMPTY_SLOTS_MANIFEST = BOX_DIR / "slots" / "empty_slots_manifest.csv"

CHOCOLATE_DIR = PROJECT_ROOT / "data" / "chocolates"
RAW_DIR = CHOCOLATE_DIR / "raw"
FLAVOR_TRAYS_DIR = RAW_DIR / "flavor_trays"
FILLED_BOXES_DIR = RAW_DIR / "filled_boxes"
CROPS_DIR = CHOCOLATE_DIR / "crops"
MANIFEST_DIR = CHOCOLATE_DIR / "manifests"

for directory in [
    FLAVOR_TRAYS_DIR,
    FILLED_BOXES_DIR,
    CROPS_DIR,
    MANIFEST_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

def find_images(directory):
    return sorted(
        path for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )

empty_slot_images = find_images(EMPTY_SLOTS_DIR)
flavor_tray_images = find_images(FLAVOR_TRAYS_DIR)
filled_box_images = find_images(FILLED_BOXES_DIR)

print("Empty-slot manifest exists:", EMPTY_SLOTS_MANIFEST.exists())
print("Empty-slot images:", len(empty_slot_images))
print("Flavor-tray images:", len(flavor_tray_images))
print("Filled-box images:", len(filled_box_images))
print("\nPlace flavor trays under:", FLAVOR_TRAYS_DIR)
print("Place filled boxes under:", FILLED_BOXES_DIR)

Empty-slot manifest exists: False
Empty-slot images: 0
Flavor-tray images: 0
Filled-box images: 0

Place flavor trays under: /homes/s959m963/Chocolathon/data/chocolates/raw/flavor_trays
Place filled boxes under: /homes/s959m963/Chocolathon/data/chocolates/raw/filled_boxes
